# Solar Filament Segmentation 2026: Original Image to H3 Semantic Probability Maps

**Reproducibility notebook (stage one), 12 November 2026**

This notebook uses only the original 180 unlabeled Kaggle TEST H-alpha JPEGs and the four released ResNet34 one-channel U-Net weights, together with the exact published native model source. It reconstructs 720 native-resolution H3 probability PNG maps, checks the SHA-256 fingerprints of original image inputs and resulting maps, and reports VRAM and execution provenance.

**Boundary:** This is a transparent stage-one replay, **not** the complete H9b Mask R-CNN / H6 / H9b crop-refiner inference chain and **not** a reproduction of the accepted CSV for Kaggle submission 58112557. No labels, annotation files or private Kaggle scores are used. Do not confuse generated map integrity with a new PQ score.

A local copy of the competition data must be obtained under the organizers' access conditions. Do not redistribute the test pixels. Use a large HDD for generated PNGs and receipts.


In [ ]:
from __future__ import annotations
import hashlib, json, os, pathlib, subprocess, sys
ROOT = pathlib.Path.cwd().resolve()
if not (ROOT / "release" / "source" / "native_model.py").is_file():
    ROOT = ROOT.parent
assert (ROOT / "release" / "source" / "native_model.py").is_file(), "Run from repository root or notebooks/"
IMAGES = pathlib.Path(os.environ["SOLARFIL_TEST_IMAGES"]).expanduser().resolve()
OUTPUT = pathlib.Path(os.environ["SOLARFIL_H3_OUTPUT_DIR"]).expanduser().resolve()
assert IMAGES.is_dir(), "SOLARFIL_TEST_IMAGES must contain competition TEST JPEG pixels"
assert OUTPUT.is_absolute() and OUTPUT != ROOT, "Set a separate large HDD output directory"
FOLDS = tuple(os.environ.get("SOLARFIL_H3_FOLDS", "F1,F2,F3,F4").split(","))
assert FOLDS and len(set(FOLDS)) == len(FOLDS) and set(FOLDS) <= {"F1","F2","F3","F4"}
DEVICE = os.environ.get("SOLARFIL_H3_DEVICE", "cuda")
assert DEVICE in ("cpu","cuda")
GPU_UUID = "GPU-b2046dc8-91f7-fd8a-b237-96bb9e293d14"
SOURCE = ROOT / "scripts" / "replay_h3_from_original_images.py"
assert SOURCE.is_file(), "Missing released source-image replay code"
print("Config validated: source", SOURCE, "folds", FOLDS, "device", DEVICE)


## Verify weights and image inventory

The executable replay program verifies each weight against the public SHA-256 manifest before model inference, checks that all 180 test filenames are present, and rejects duplicate image stems. The source model uses the same 2048-native solar-disk normalization and D4 eight-view transforms as the original H3 pipeline.


In [ ]:
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for data in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(data)
    return h.hexdigest()

entries = {}
for line in (ROOT / "release/model_metadata/WEIGHTS_SHA256SUMS.txt").read_text().splitlines():
    digest, filename = line.split()
    assert len(digest) == 64
    assert filename not in entries
    entries[filename] = digest
for fold in FOLDS:
    name = f"{fold}_resnet34_s0_fp16.pt"
    assert sha256_file(ROOT / "release/models" / name) == entries[name], name
files = sorted(IMAGES.glob("*.jpeg"))
assert len(files) == 180 and len({x.stem for x in files}) == 180
print("CHECKPOINTS_VERIFIED", len(FOLDS), "ORIGINAL_TEST_IMAGES", len(files))


## Execute stage-one inference

CUDA is restricted to the assigned physical GPU0 and the program refuses to start if another CUDA process holds it. GPU1 remains unallocated by this workflow. Stage outputs are written atomically to the user-selected HDD directory. For a genuinely interrupted run, set environment `SOLARFIL_H3_RESUME=1` to reuse validated existing maps. Completed receipts are not overwritten. This cell intentionally never submits anything to Kaggle.


In [ ]:
env = os.environ.copy()
if DEVICE == "cuda":
    env["CUDA_VISIBLE_DEVICES"] = GPU_UUID
env.setdefault("OMP_NUM_THREADS", "2")
env.setdefault("OPENBLAS_NUM_THREADS", "2")
for fold in FOLDS:
    receipt = OUTPUT / f"{fold}_h3_replay_receipt.json"
    if receipt.is_file():
        verified = json.loads(receipt.read_text())
        assert verified["full_180_images"] and verified["weight_sha256"] == entries[f"{fold}_resnet34_s0_fp16.pt"]
        print("EXISTING_COMPLETE_RECEIPT", fold)
        continue
    cmd = [
        sys.executable, "-u", str(SOURCE), "--images", str(IMAGES),
        "--out", str(OUTPUT), "--fold", fold, "--device", DEVICE,
    ]
    if env.get("SOLARFIL_H3_RESUME") == "1":
        cmd.append("--resume")
    subprocess.run(cmd, check=True, env=env, cwd=ROOT)


## Read-only output verification

Every receipt binds the original H-alpha input image hashes to the generated 2048-native PNGs, the exact model source hash, the public model checkpoint SHA, and the GPU runtime. This validation does not read test labels, any annotation files, or the Kaggle leaderboard.


In [ ]:
import cv2
for fold in FOLDS:
    receipt = json.loads((OUTPUT / f"{fold}_h3_replay_receipt.json").read_text())
    assert receipt["status"] == "H3_STAGE_ONE_REPLAY_COMPLETE_NOT_FINAL_CSV"
    assert receipt["full_180_images"] and receipt["images_processed"] == 180
    assert receipt["test_labels_read"] is False
    assert receipt["end_to_end_submission_csv_reproduced"] is False
    assert receipt["weight_sha256"] == entries[f"{fold}_resnet34_s0_fp16.pt"]
    for row in receipt["image_sha256"]:
        assert sha256_file(IMAGES / row["file"]) == row["input_sha256"]
        output = OUTPUT / fold / (pathlib.Path(row["file"]).stem + ".png")
        assert sha256_file(output) == row["map_sha256"]
        arr = cv2.imread(str(output), cv2.IMREAD_GRAYSCALE)
        assert arr is not None and arr.dtype.name == "uint8" and arr.shape == (2048,2048)
    print("STAGE_ONE_MAPS_INTEGRITY_VERIFIED", fold, 180)


## Scope of the result

This notebook is source-to-H3-map reproducibility, with public weights and no data leakage. It does not verify historical *bit-exact* PNG parity, detector proposal reconstruction, H6 instance selection, crop refinement, the original final `submission_h9b_nosite.csv`, the public 0.39 PQ, or any required Google Form receipt. Those stages require separate evidence. Do not upload the H3 maps as a CSV or claim a new leaderboard score.
